> **ARCHIVE — superseded.** This notebook is kept as a record of the research path, not as working code. Credentials were removed and outputs cleared. Results in it may rest on defects documented in `docs/RESEARCH_LOG.md` (stage: v1 audit, point-in-time on survivors). Comments are partly in Vietnamese. The maintained implementation is `src/memescan/`.

# v1 Lean — Second Try: thay ro HINDSIGHT bang ro POINT-IN-TIME

Giu **nguyen xi** thuat toan v1 (entry/exit/portfolio khong doi mot dong).
Chi thay **cach dung ro co phieu**:

| | v1 goc | v1 second try |
|---|---|---|
| Ro | 82 ma co dinh, chon nam 2026 khi da biet ket qua | Cap nhat MOI NGAY, chi nhin lich su truoc t |
| Hop le? | Khong (hindsight) | Co |

**Ro tai ngay t = A ∪ B**, ca hai deu tinh tu du lieu <= t-1:
- **A** — dang o duoi 50% dinh moi thoi dai (ATH tinh den t-1)
- **B** — da tung co dot meme KET THUC truoc t (dot bat dau ngay d chi "biet" duoc o ngay dinh,
  vi phai thay gia chay du 50%; nen dung **peak_date < t** lam moc, khong dung ignition_date)

Sau do so 3 ban: **hindsight (doi chieu)** / **PIT** / **PIT + social**.

In [ ]:
# Cell 1 - Load + tham so v1 (KHONG DOI so voi v1 goc)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

C = Path("meme_cache"); ALLOC, CAP, MAXPOS = 10000.0, 100000.0, 10
V1 = dict(rvol_min=2.0, ext_cap=0.50, mom5_min=0.05,     # entry - y het v1
          hard_stop=0.08, atr_mult=3.0, ma_exit=10, max_hold=60,
          early_days=2, early_mfe=0.03)                   # exit - y het v1

pool = pd.read_parquet(C/"pool_universe.parquet")
seed82 = set(pd.read_parquet(C/"bt_daily.parquet").ticker.unique())   # ro hindsight, chi de DOI CHIEU
cl = pool.pivot(index="date", columns="ticker", values="close")
vo = pool.pivot(index="date", columns="ticker", values="vol")
hi = pool.pivot(index="date", columns="ticker", values="high")
lo = pool.pivot(index="date", columns="ticker", values="low")
op = pool.pivot(index="date", columns="ticker", values="open")
dv = cl*vo; DATES = list(cl.index); TICKS = list(cl.columns)
print(f"Pool {len(TICKS)} ma | {len(DATES)} phien | {DATES[0].date()} -> {DATES[-1].date()}")
print(f"Ro hindsight (chi doi chieu): {len(seed82 & set(TICKS))} ma")

In [ ]:
# Cell 2 - RO POINT-IN-TIME: A = duoi 50% ATH, B = da tung la meme (dot da xong truoc t)
cl_s = cl.shift(1)                                     # moi thu chi dung <= t-1

# --- A: duoi 50% dinh moi thoi dai (cummax = chi dinh trong QUA KHU) ---
A = (cl_s/cl_s.cummax() - 1) < -0.50

# --- B: da tung co dot meme ket thuc truoc t ---
B = pd.DataFrame(False, index=DATES, columns=TICKS)
n_ev = 0
for sym in TICKS:
    c = cl[sym].values; n = len(c); t = 0; peaks = []
    while t < n-1:
        if np.isnan(c[t]): t += 1; continue
        w = c[t+1:min(t+21, n)]; w = w[~np.isnan(w)]
        if len(w) == 0: t += 1; continue
        if w.max()/c[t]-1 >= 0.50:                      # dot meme = chay >=50% trong 20 phien
            seg = c[t+1:min(t+21, n)]
            pk = t+1+int(np.nanargmax(seg))
            peaks.append(pk); n_ev += 1; t = pk+1
        else: t += 1
    if peaks:
        first_known = peaks[0]+1                        # chi BIET tu sau ngay dinh -> +1
        if first_known < n: B.iloc[first_known:, TICKS.index(sym)] = True

UNIV_PIT = (A | B) & (cl_s > 1) & (dv.shift(1).rolling(20).mean() >= 1e6)
UNIV_HIND = pd.DataFrame(False, index=DATES, columns=TICKS)
for s in seed82 & set(TICKS): UNIV_HIND[s] = True

print(f"Phat hien {n_ev} dot meme -> {int(B.any().sum())} ma tung la meme")
print(f"\nKich thuoc ro theo thoi gian (PIT tu lon dan vi lich su tich luy):")
for y in sorted({d.year for d in DATES}):
    m = [i for i,d in enumerate(DATES) if d.year == y]
    a = A.iloc[m].sum(axis=1).mean(); b = B.iloc[m].sum(axis=1).mean(); u = UNIV_PIT.iloc[m].sum(axis=1).mean()
    print(f"  {y}: duoi-50%-ATH {a:5.0f} | tung-la-meme {b:5.0f} | HOP NHAT (sau loc thanh khoan) {u:5.0f} ma/ngay")
print(f"\nTB ca ky: ro PIT {UNIV_PIT.sum(axis=1).mean():.0f} ma/ngay vs ro hindsight {len(seed82 & set(TICKS))} ma co dinh")

In [ ]:
# Cell 3 - Engine v1 (copy nguyen logic entry/exit, chi doi cach kiem tra "co trong ro")
def _atr(h, l, c, w=14):
    n = len(c); t = np.zeros(n)
    for i in range(1, n): t[i] = max(h[i]-l[i], abs(h[i]-c[i-1]), abs(l[i]-c[i-1]))
    a = np.full(n, np.nan)
    for i in range(w, n): a[i] = t[i-w+1:i+1].mean()
    return a
ATRc = {s: _atr(np.nan_to_num(hi[s].values), np.nan_to_num(lo[s].values), np.nan_to_num(cl[s].values))
        for s in TICKS}

def run_v1(in_universe, rank=None):
    """in_universe(date, sym) -> bool. rank: DataFrame diem de uu tien khi qua nhieu tin hieu (tuy chon).
       Entry: quyet dinh tai close[i], VAO tai open[i+1]. Stop khop trong phien. -> khong look-ahead."""
    sigs = defaultdict(list)
    for sym in TICKS:
        c = cl[sym].values; v = vo[sym].values; h = hi[sym].values
        l = lo[sym].values; o = op[sym].values
        if (~np.isnan(c)).sum() < 200: continue
        for i in range(21, len(c)-2):
            if np.isnan(c[i]) or np.isnan(o[i+1]): continue
            d = DATES[i]
            if not in_universe(d, sym): continue
            b = np.nanmean(v[i-20:i]); rv = v[i]/b if b > 0 else 0
            ma = np.nanmean(c[i-20:i]); ext = c[i]/ma-1 if ma > 0 else 0
            typ = (h[i]+l[i]+c[i])/3
            if (rv >= V1["rvol_min"] and c[i] > typ and c[i] > c[i-1] and c[i] > ma
                    and ext <= V1["ext_cap"] and c[i]/c[i-5]-1 > V1["mom5_min"]):
                sc = float(rank.at[d, sym]) if rank is not None else rv
                sigs[d].append((sc, sym, i))
    trades, open_until = [], []
    for d in DATES:
        if d not in sigs: continue
        for sc, sym, i in sorted(sigs[d], reverse=True):
            open_until = [x for x in open_until if x > d]
            if len(open_until) >= MAXPOS: break
            c = cl[sym].values; o = op[sym].values; l = lo[sym].values; atr = ATRc[sym]
            ei = i+1; e = o[i+1]
            if e <= 0: continue
            peak = e; xi = min(ei+V1["max_hold"], len(c)-1); ex = None; why = "max_hold"
            for j in range(ei+1, xi+1):
                if np.isnan(c[j]): continue
                sp = e*(1-V1["hard_stop"])
                if o[j] <= sp: xi, ex, why = j, o[j], "stop_gap"; break
                if l[j] <= sp: xi, ex, why = j, sp, "stop"; break
                if (j-ei) >= V1["early_days"] and (np.nanmax(c[ei+1:j+1])/e-1) < V1["early_mfe"]:
                    xi, ex, why = j, c[j], "early_exit"; break
                peak = max(peak, c[j])
                if c[j] <= peak - V1["atr_mult"]*(atr[j] if not np.isnan(atr[j]) else 0):
                    xi, ex, why = j, c[j], "atr_trail"; break
                if c[j] > e and c[j] < np.nanmean(c[max(0, j-V1["ma_exit"]+1):j+1]):
                    xi, ex, why = j, c[j], "ma_break"; break
            if ex is None or np.isnan(ex): ex = c[xi]
            open_until.append(DATES[xi])
            trades.append({"ticker": sym, "ei": ei, "xi": xi, "entry_date": DATES[ei],
                           "exit_date": DATES[xi], "entry_px": e, "exit_px": ex,
                           "ret": ex/e-1, "days": xi-ei, "exit_reason": why})
    tk = pd.DataFrame(trades)
    if len(tk) == 0: return None, None
    tk["pnl"] = ALLOC*tk["ret"]
    acc = defaultdict(float)
    for _, t in tk.iterrows():
        c = cl[t["ticker"]].values; sh = ALLOC/t["entry_px"]
        for j in range(int(t["ei"])+1, int(t["xi"])+1):
            if not np.isnan(c[j]) and not np.isnan(c[j-1]): acc[DATES[j]] += sh*(c[j]-c[j-1])
    eq = CAP + pd.Series([acc.get(d, 0.) for d in DATES], index=DATES).cumsum()
    return tk, eq

def stats(tk, eq):
    r = eq.pct_change().dropna(); yrs = (DATES[-1]-DATES[0]).days/365.25
    dn = r[r < 0]
    return dict(n=len(tk), win=(tk.ret > 0).mean(), exp=tk.ret.mean(),
                cagr=(eq.iloc[-1]/CAP)**(1/yrs)-1,
                sharpe=r.mean()/r.std()*np.sqrt(252) if r.std() > 0 else np.nan,
                sortino=r.mean()/dn.std()*np.sqrt(252) if len(dn) and dn.std() > 0 else np.nan,
                maxdd=((eq-eq.cummax())/eq.cummax()).min(), final=eq.iloc[-1])
print("Engine v1 san sang.")

In [ ]:
# Cell 4 - Social (tuy chon) - dung lam diem UU TIEN khi co nhieu tin hieu cung ngay
def align_daily(df, tgt):
    d = df.copy(); d.index = pd.DatetimeIndex(d.index).normalize()
    d = d[~d.index.duplicated(keep="last")]
    o = d.reindex(pd.DatetimeIndex(tgt).normalize()); o.index = tgt
    return o
zs = lambda d: d.sub(d.mean(axis=1), axis=0).div(d.std(axis=1).replace(0, np.nan), axis=0)

SOC_RANK = None
rm = None
try: rm = pd.read_parquet(C/"reddit_mentions.parquet")
except Exception: pass
if rm is not None and len(rm):
    rm = rm[rm["ticker"].isin(TICKS)]
    M = align_daily(rm.pivot_table(index="date", columns="ticker", values="mentions", aggfunc="sum"),
                    DATES).reindex(columns=TICKS)
    cover = M.notna().any(axis=1); M = M.fillna(0.0)
    try: words = {w.strip().upper() for w in open("/usr/share/dict/words") if 2 <= len(w.strip()) <= 5}
    except Exception: words = set()
    Mc = M.loc[cover]
    burst = {t: (Mc[t].max()/max(Mc[t][Mc[t] > 0].median(), 1)) if (Mc[t] > 0).any() else 0 for t in TICKS}
    BLOCK = [t for t in TICKS if t in words and burst[t] < 10 and (Mc[t] > 0).mean() > .6]
    M[BLOCK] = 0.0
    Ms = np.log1p(M).shift(1)                     # tre 1 ngay
    # IC do duoc: accel20 = +0.025 (DUONG) | level = -0.054 (AM: da dong nguoi = vao muon)
    ACC = zs(Ms - Ms.rolling(20, min_periods=5).mean()).fillna(0)
    LVL = zs(Ms.rolling(20, min_periods=3).mean()).fillna(0)
    SOC_RANK = 1.5*ACC - 1.0*LVL
    print(f"Social: {int(cover.sum())}/{len(DATES)} phien ({cover.sum()/len(DATES):.0%}) | chan {len(BLOCK)} ma nhiem")
else:
    print("Social: chua co du lieu -> bo qua ban co social")

In [ ]:
# Cell 5 - Chay 3 ban va so sanh
PIT = lambda d, s: bool(UNIV_PIT.at[d, s])
HND = lambda d, s: bool(UNIV_HIND.at[d, s])

runs = {}
tk, eq = run_v1(HND);  runs["1. RO HINDSIGHT 82 ma (an gian - doi chieu)"] = (tk, eq)
tk, eq = run_v1(PIT);  runs["2. RO POINT-IN-TIME (hop le)"] = (tk, eq)
if SOC_RANK is not None:
    tk, eq = run_v1(PIT, rank=SOC_RANK); runs["3. RO PIT + social uu tien"] = (tk, eq)

rows = {k: stats(t, e) for k, (t, e) in runs.items() if t is not None}
out = pd.DataFrame(rows).T[["n", "win", "exp", "cagr", "sharpe", "sortino", "maxdd", "final"]]
print(out.to_string(formatters={"win": "{:.1%}".format, "exp": "{:+.2%}".format, "cagr": "{:.1%}".format,
      "sharpe": "{:.2f}".format, "sortino": "{:.2f}".format, "maxdd": "{:.1%}".format,
      "final": "${:,.0f}".format}))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for k, (t, e) in runs.items():
    if e is None: continue
    ls = "--" if "HINDSIGHT" in k else "-"
    ax[0].plot(e.index, e.values, ls, lw=1.6, label=f"{k[:28]} (${e.iloc[-1]:,.0f})")
ax[0].axhline(CAP, c="gray", lw=.8); ax[0].set_yscale("log")
ax[0].set_title("Equity (log)"); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
ax[1].plot(DATES, UNIV_PIT.sum(axis=1).values, lw=1.2, label="ro PIT")
ax[1].axhline(len(seed82 & set(TICKS)), c="crimson", ls="--", lw=1, label="ro hindsight (co dinh)")
ax[1].set_title("Kich thuoc ro theo thoi gian"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 6 - Chi tiet ban PIT: exit reason, theo nam, top trade
key = "2. RO POINT-IN-TIME (hop le)"
tk, eq = runs[key]
if tk is not None:
    print("Exit reason:")
    print(tk.groupby("exit_reason").agg(n=("ret","size"), win=("ret", lambda x:(x>0).mean()),
          avg=("ret","mean"), pnl=("pnl","sum")).round(3).to_string())
    print("\nTheo nam:")
    print(tk.assign(y=pd.to_datetime(tk.entry_date).dt.year).groupby("y").agg(
          n=("ret","size"), win=("ret", lambda x:(x>0).mean()), pnl=("pnl","sum")).round(3).to_string())
    print("\nTop 10 lai:")
    print(tk.nlargest(10,"ret")[["ticker","entry_date","entry_px","exit_px","ret","days","exit_reason"]]
          .assign(entry_date=lambda x: pd.to_datetime(x.entry_date).dt.date).to_string(index=False))
    print(f"\nTop 10 trade dong gop {tk.nlargest(10,'pnl').pnl.sum()/tk.pnl.sum():.0%} tong P&L")
    tk.to_csv(C/"v1_pit_trades.csv", index=False); print("Saved: meme_cache/v1_pit_trades.csv")